# Financial News NER & Large-Cap Relevance (Interest) Classification
### Powered by Google Gemini Flash API

This notebook replaces the legacy Claude notebook to perform automated Named Entity Recognition (NER) and binary interest classification on our financial news dataset (2024–2026).

### Objectives:
1. **Company Extraction (`Company`)**: Identify specific companies mentioned in headlines (e.g., Reliance Industries, TCS, HDFC Bank, Infosys), or `None` if broad market/macro.
2. **Sector Classification (`Sector`)**: Map to standardized industry sectors (Energy, Banking, Information Technology, Automotive, Pharmaceuticals, FMCG, Macro/Commodities, etc.).
3. **Large-Cap Interest Filtering (`Interest`)**:
   - **`1` (Relevant)**: Direct large-cap companies (Nifty 50 / BSE Sensex heavyweights) or major macroeconomic market catalysts (Crude Oil, Gold, RBI monetary policy, Inflation, Union Budget, FII flows).
   - **`0` (Irrelevant)**: Small/micro-cap penny stocks, gossip, non-commercial news, or noise.

### Architecture:
- **Model**: Google Gemini Flash (`gemini-3.5-flash-lite` / `gemini-3.8-flash`) with structured JSON schema enforcement.
- **High-Throughput Batching**: Groups 15–20 headlines per request for speed and optimal token utilization.
- **Fail-Safe Checkpointing**: Progress is incrementally auto-saved to disk so runs can be safely paused and resumed anytime.

In [1]:
import os
import sys
import json
import time
import requests
import pandas as pd
from typing import List, Dict, Any
from tqdm.auto import tqdm

# Gemini API Configuration
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY_HERE")

# Primary and fallback models for maximum reliability
MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash",
    "gemini-flash-latest"
]

print(f"[Setup] API key loaded (prefix: {GEMINI_API_KEY[:6]}...)")
print(f"[Setup] Configured models with automatic fallback: {MODELS}")

[Setup] API key loaded (prefix: AQ.Ab8...)
[Setup] Configured models with automatic fallback: ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-3.8-flash', 'gemini-flash-latest']


/home/arnav/stock_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Scraped Financial News Dataset (2024–2026)

In [ ]:
# Determine project root path
project_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.exists(os.path.join(os.getcwd(), "..")) else os.getcwd()
dataset_path = os.path.join(project_root, "Datasets", "scraped_financial_news.csv")

if not os.path.exists(dataset_path):
    # Fallback to local path if running from root
    dataset_path = os.path.join(os.getcwd(), "Datasets", "scraped_financial_news.csv")

print(f"Loading dataset from: {dataset_path}")
df = pd.read_csv(dataset_path)
print(f"Dataset Shape: {df.shape}")
print(f"Total Headlines: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Gemini API Client with Structured JSON & Resilience

In [ ]:
def call_gemini_api(prompt: str, max_retries: int = 5) -> str:
    """
    Calls the Google Gemini API with structured JSON output and automatic
    fallback across flash models on 503 high-demand or 429 rate limit spikes.
    """
    payload = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {
            "responseMimeType": "application/json",
            "temperature": 0.0
        }
    }

    headers = {"Content-Type": "application/json"}

    for attempt in range(max_retries):
        for model_name in MODELS:
            url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={GEMINI_API_KEY}"
            try:
                resp = requests.post(url, json=payload, headers=headers, timeout=25)
                if resp.status_code == 200:
                    data = resp.json()
                    return data["candidates"][0]["content"]["parts"][0]["text"]
                elif resp.status_code == 429:
                    time.sleep(2.0 * (attempt + 1))
                elif resp.status_code == 503:
                    # Model busy, try next model in fallback list
                    continue
                else:
                    time.sleep(1.0)
            except requests.RequestException:
                time.sleep(1.0)
        time.sleep(1.5 * (attempt + 1))

    raise RuntimeError(f"Gemini API request failed after {max_retries} retries across all fallback models.")

## 3. Interactive Single Headline Test
Verify prompt instructions on individual sample headlines.

In [ ]:
def analyze_single_headline(headline: str) -> Dict[str, Any]:
    """
    Extracts Company, Sector, and Interest (1 for large cap/macro, 0 otherwise)
    for an individual news headline.
    """
    prompt = f"""You are an expert Indian financial market analyst.
Analyze this news headline and identify:
1. "Company": The specific company name (e.g. "Reliance Industries", "TCS", "HDFC Bank", "State Bank of India"), or "None" if general macroeconomic or market-wide news.
2. "Sector": The primary industry sector (e.g. "Energy", "Banking", "Information Technology", "Automotive", "Pharmaceuticals", "Macro / Economy", "None").
3. "Interest": Binary integer (1 or 0).
   - Set to 1 if the headline is relevant to Indian large-cap companies (NIFTY 50 / Sensex heavyweights) or key macroeconomic market catalysts (Crude Oil, Gold, Inflation, RBI repo rate, Budget, FII flows).
   - Set to 0 if it is a penny/micro-cap stock, Bollywood/sports gossip, non-commercial news, or irrelevant noise.

Headline: "{headline}"

Respond strictly with valid JSON conforming to this schema:
{{"Company": string, "Sector": string, "Interest": integer}}"""

    raw_json = call_gemini_api(prompt)
    return json.loads(raw_json)

# Test on 3 diverse sample headlines
test_samples = [
    "Reliance Industries Q1 net profit jumps 15% on oil-to-chemicals rebound",
    "Crude oil prices drop as OPEC weighs production increase, relieving Indian import bill",
    "Small penny stock XYZ Textiles hits lower circuit after promoter sells shares"
]

for h in test_samples:
    res = analyze_single_headline(h)
    print(f"Headline: {h}")
    print(f" -> Company:  {res.get('Company')}")
    print(f" -> Sector:   {res.get('Sector')}")
    print(f" -> Interest: {res.get('Interest')}")
    print("-" * 60)

## 4. Batch Processing Pipeline with Incremental Checkpoints
Processes headlines in batches of 15–20 to achieve high throughput while maintaining checkpoints on disk.

In [ ]:
def analyze_headline_batch(batch: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Processes a batch of headlines in a single Gemini API call with structured JSON output.
    """
    prompt = f"""You are an expert financial market analyst specializing in Indian equities.
For each news item below, extract:
1. "Company": Name of the primary company mentioned, or "None" if general macro/market news.
2. "Sector": Primary industry sector (Energy, Banking, Information Technology, Automotive, Pharmaceuticals, FMCG, Macro / Economy, etc.), or "None".
3. "Interest": Binary integer (1 or 0).
   - Set to 1 if relevant to Indian large-cap companies (Nifty 50, BSE 100, blue chips) OR key macroeconomic market drivers (Crude Oil, Gold, RBI, Inflation, Union Budget, FII flows).
   - Set to 0 if penny/micro-cap, irrelevant, or non-commercial.

Input Items:
{json.dumps(batch, indent=2)}

Respond ONLY with a JSON array conforming to this schema:
[
  {{"id": integer, "Company": string, "Sector": string, "Interest": integer}},
  ...
]"""

    raw_json = call_gemini_api(prompt)
    try:
        return json.loads(raw_json)
    except json.JSONDecodeError:
        # Clean possible markdown formatting ticks
        clean = raw_json.strip().removeprefix("```json").removesuffix("```").strip()
        return json.loads(clean)


def run_ner_pipeline(df: pd.DataFrame, batch_size: int = 15, max_records: int = None) -> pd.DataFrame:
    """
    Runs batch NER and Interest classification with checkpoint saving.
    """
    checkpoint_file = os.path.join(project_root, "Datasets", "scraped_financial_news_ner_checkpoint.csv")
    annotated_records = {}

    # Load existing checkpoint if available
    if os.path.exists(checkpoint_file):
        chk_df = pd.read_csv(checkpoint_file)
        for _, r in chk_df.iterrows():
            if pd.notna(r.get("URL")) and pd.notna(r.get("Interest")):
                annotated_records[r["URL"]] = {
                    "Company": r.get("Company", "None"),
                    "Sector": r.get("Sector", "None"),
                    "Interest": int(r.get("Interest", 0))
                }
        print(f"[Resume] Loaded {len(annotated_records)} existing annotations from checkpoint.")

    # Filter rows that still need processing
    records_to_process = []
    for idx, row in df.iterrows():
        u = str(row.get("URL", "")).strip()
        if u not in annotated_records:
            records_to_process.append({
                "row_index": idx,
                "url": u,
                "headline": str(row.get("Headlines", "")).strip()
            })

    if max_records:
        records_to_process = records_to_process[:max_records]

    print(f"[Pipeline] Remaining headlines to annotate: {len(records_to_process):,} (Batch Size: {batch_size})")

    # Process in batches
    for i in tqdm(range(0, len(records_to_process), batch_size), desc="Annotating with Gemini Flash"):
        batch_items = records_to_process[i : i + batch_size]
        query_batch = [{"id": item["row_index"], "headline": item["headline"]} for item in batch_items]

        try:
            results = analyze_headline_batch(query_batch)
            id_to_res = {r["id"]: r for r in results if "id" in r}

            for item in batch_items:
                idx = item["row_index"]
                res = id_to_res.get(idx, {})
                annotated_records[item["url"]] = {
                    "Company": res.get("Company", "None"),
                    "Sector": res.get("Sector", "None"),
                    "Interest": int(res.get("Interest", 0))
                }
        except Exception as e:
            print(f"\n[Warning] Batch at index {i} failed: {e}. Retrying item-by-item...")
            for item in batch_items:
                try:
                    single_res = analyze_single_headline(item["headline"])
                    annotated_records[item["url"]] = {
                        "Company": single_res.get("Company", "None"),
                        "Sector": single_res.get("Sector", "None"),
                        "Interest": int(single_res.get("Interest", 0))
                    }
                except Exception as inner_e:
                    annotated_records[item["url"]] = {"Company": "None", "Sector": "None", "Interest": 0}

        # Save checkpoint every 10 batches
        if (i // batch_size) % 10 == 0 or (i + batch_size >= len(records_to_process)):
            chk_rows = []
            for _, row in df.iterrows():
                u = str(row.get("URL", "")).strip()
                if u in annotated_records:
                    r_dict = row.to_dict()
                    r_dict.update(annotated_records[u])
                    chk_rows.append(r_dict)
            if chk_rows:
                pd.DataFrame(chk_rows).to_csv(checkpoint_file, index=False)

        time.sleep(0.5)  # Safe delay between batches

    # Merge annotations with main dataframe
    df_annotated = df.copy()
    df_annotated["Company"] = df_annotated["URL"].apply(lambda u: annotated_records.get(str(u).strip(), {}).get("Company", "None"))
    df_annotated["Sector"] = df_annotated["URL"].apply(lambda u: annotated_records.get(str(u).strip(), {}).get("Sector", "None"))
    df_annotated["Interest"] = df_annotated["URL"].apply(lambda u: annotated_records.get(str(u).strip(), {}).get("Interest", 0))

    return df_annotated

## 5. Execute Pipeline & Verify Results
Run on a test slice (or the entire dataset), then inspect classification distributions.

In [ ]:
# Example: Run pipeline on first 100 records for immediate verification
# To run on all 10k+ records, set max_records=None
df_sample_annotated = run_ner_pipeline(df, batch_size=20, max_records=100)

print("\n--- Verification Metrics ---")
print("Interest Distribution (1 = Large-Cap/Macro, 0 = Other):")
print(df_sample_annotated["Interest"].value_counts())

print("\nTop Identified Companies:")
print(df_sample_annotated["Company"].value_counts().head(10))

print("\nSector Distribution:")
print(df_sample_annotated["Sector"].value_counts().head(10))

df_sample_annotated[["Headlines", "Company", "Sector", "Interest"]].head(15)

## 6. Export Final Annotated Dataset
Save the enriched dataset to CSV, JSON, and JSONL formats for downstream sentiment modeling and classification.

In [ ]:
output_csv = os.path.join(project_root, "Datasets", "financial_news_ner_annotated.csv")
output_json = os.path.join(project_root, "Datasets", "financial_news_ner_annotated.json")
output_jsonl = os.path.join(project_root, "Datasets", "financial_news_ner_annotated.jsonl")

# Export
df_sample_annotated.to_csv(output_csv, index=False)
df_sample_annotated.to_json(output_json, orient="records", indent=2, force_ascii=False)
df_sample_annotated.to_json(output_jsonl, orient="records", lines=True, force_ascii=False)

print(f"Successfully saved annotated dataset to:")
print(f"- CSV:   {output_csv}")
print(f"- JSON:  {output_json}")
print(f"- JSONL: {output_jsonl}")